In [1]:
!pip install requests folium shapely lxml numpy -q


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import requests
import folium
import json
import zipfile
import io
import sys
import math
import time
from datetime import datetime
from lxml import etree
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import nearest_points, unary_union
import pandas as pd

In [28]:
URL = "https://www.waze.com/row-partnerhub-api/partners/12221294397/waze-feeds/5b6ed9eb-a68d-4ed5-a4a6-585593c8a8e6?format=1"

data = requests.get(URL, timeout=30).json()

df = pd.json_normalize(data["alerts"])

# Separar location.x / location.y en columnas limpias
df.rename(columns={"location.x": "longitude", "location.y": "latitude"}, inplace=True)

# Convertir timestamp a fecha legible
df["fecha"] = pd.to_datetime(df["pubMillis"], unit="ms")

print(df.shape)
df.head()

(46, 17)


,country,city,reportRating,reportByMunicipalityUser,confidence,reliability,type,uuid,roadType,magvar,subtype,street,reportDescription,pubMillis,longitude,latitude,fecha
0,CO,Cartagena de Indias,5,false,0,6,ROAD_CLOSED,fa4944aa-3dfb-4118-b061-e16b0d3176f4,1,287,,Calle 3 B,,1773207594000,-75.483975,10.371051,2026-03-11 05:39:54
1,CO,Cartagena de Indias,0,false,1,7,HAZARD,b6bc1366-7dd4-422e-a924-356edbd2e756,1,292,HAZARD_ON_ROAD_POT_HOLE,Calle 32 C,NaN,1777086871000,-75.515236,10.412009,2026-04-25 03:14:31
2,CO,Cartagena de Indias,1,false,5,10,HAZARD,5112c10e-1d51-4f46-84c0-1e17d64c59de,1,118,HAZARD_ON_ROAD_POT_HOLE,Calle 29,NaN,1777584324000,-75.515342,10.406958,2026-04-30 21:25:24
3,CO,Cartagena de Indias,1,false,0,7,HAZARD,798f004c-e312-4dcd-bc5f-5e95e25546af,1,164,HAZARD_ON_ROAD_POT_HOLE,Diagonal 22,NaN,1777584170000,-75.520835,10.409017,2026-04-30 21:22:50
4,CO,Cartagena de Indias,3,false,0,5,HAZARD,dd5cb221-366d-4c26-a17c-18494911334a,1,274,HAZARD_ON_ROAD_POT_HOLE,Transversal 52,NaN,1777589291000,-75.519640,10.391878,2026-04-30 22:48:11


In [15]:
data["alerts"]

[{'country': 'CO',
  'city': 'Cartagena de Indias',
  'reportRating': 5,
  'reportByMunicipalityUser': 'false',
  'confidence': 0,
  'reliability': 6,
  'type': 'ROAD_CLOSED',
  'uuid': 'fa4944aa-3dfb-4118-b061-e16b0d3176f4',
  'roadType': 1,
  'magvar': 287,
  'subtype': '',
  'street': 'Calle 3 B',
  'reportDescription': '',
  'location': {'x': -75.483975, 'y': 10.371051},
  'pubMillis': 1773207594000},
 {'country': 'CO',
  'city': 'Cartagena de Indias',
  'reportRating': 0,
  'reportByMunicipalityUser': 'false',
  'confidence': 1,
  'reliability': 7,
  'type': 'HAZARD',
  'uuid': 'b6bc1366-7dd4-422e-a924-356edbd2e756',
  'roadType': 1,
  'magvar': 292,
  'subtype': 'HAZARD_ON_ROAD_POT_HOLE',
  'street': 'Calle 32 C',
  'location': {'x': -75.515236, 'y': 10.412009},
  'pubMillis': 1777086871000},
 {'country': 'CO',
  'city': 'Cartagena de Indias',
  'reportRating': 1,
  'reportByMunicipalityUser': 'false',
  'confidence': 1,
  'reliability': 9,
  'type': 'HAZARD',
  'uuid': '55694457

In [10]:
CONFIG = {
    # Centro de búsqueda (lat, lon)
    "center_lat":  10.39972,
    "center_lon": -75.51444,
 
    # Radio de búsqueda en km
    "search_radius_km": 10,
 
    # Radio de área de influencia en METROS (editable en el mapa)
    "buffer_radius_m": 300,
 
    # Ruta al archivo KMZ (puede ser relativa o absoluta)
    "kmz_path": "/workspaces/Rup_tub/KMZ/TuberiaAcero_Cartagena.kmz",
 
    # Nombre del archivo de salida
    "output_html": "mapa_waze_hazards.html",
 
    # Usar mock data si la API de Waze no responde (True/False)
    "use_mock_fallback": True,
 
    # Modo región: "row" = Rest of World (Latam), "us" = USA
    "waze_region": "row",
}

In [5]:
# ─────────────────────────────────────────────
# 1.  WAZE API  (endpoint no-oficial público)
# ─────────────────────────────────────────────
API_KEY = "ak_29d1gn2u8dtu3va7l24ap0qj7inshdvuvgeycj8drp96jxy"


url = "https://api.openwebninja.com/waze/alerts-and-jams"
 

WAZE_HEADERS = {
     "x-api-key": API_KEY
}

In [11]:
import json
from datetime import datetime

def save_response(data, prefix="waze_data"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prefix}_{timestamp}.json"
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"[WAZE] ✓ Guardado en: {filename}")
    return filename

In [7]:
def fetch_waze_alerts(CENTER, radius_km, url):
    """
    Consulta la API no-oficial de Waze.
    Retorna la lista cruda de alertas o None si falla.
 
    Nota: Waze CCP (Connected Citizens Program) ofrece acceso
    oficial con más datos. Para escalar, reemplazar aquí.
    """

    params = {
        "max_jams":     0,
        "max_alerts":  200,
        "center":   CENTER,
        "radius":   radius_km
        #"alert_types": "ROAD_CLOSED, WEATHERHAZARD"
    }
 
    print(f"\n[WAZE] Consultando: {url}")
    print(f"[WAZE] BBox: center={CENTER} con radio={radius_km:.4f}")
 
    try:
        r = requests.get(url, params=params, headers=WAZE_HEADERS, timeout=12)
        r.raise_for_status()
        data = r.json()
        total = len(data.get("data", {}).get("alerts", []))
        print(f"[WAZE] ✓ Respuesta OK — {total} alertas totales recibidas")
        return data
    except requests.exceptions.Timeout:
        print("[WAZE] ✗ Timeout — Waze no respondió a tiempo")
    except requests.exceptions.HTTPError as e:
        print(f"[WAZE] ✗ HTTP Error: {e}")
    except Exception as e:
        print(f"[WAZE] ✗ Error: {e}")
    return None

In [8]:
TARGET_TYPES = {"HAZARD", "ROAD_CLOSED"}

TARGET_SUBTYPES = {
    None,
    "HAZARD_ON_ROAD_CONSTRUCTION",
    "ROAD_CLOSED_CONSTRUCTION"
}

def filter_construction_hazards(waze_data):

    if not waze_data:
        return []

    all_alerts = waze_data.get("data", {}).get("alerts", [])

    filtered = [
        a for a in all_alerts
        if a.get("type") in TARGET_TYPES
        and a.get("subtype") in TARGET_SUBTYPES
    ]

    print(
        f"\n[FILTRO] {len(all_alerts)} alertas totales → {len(filtered)} eventos de construcción"
    )
    return filtered

In [12]:
KML_NS = {
    "kml":  "http://www.opengis.net/kml/2.2",
    "kml1": "http://earth.google.com/kml/2.0",
    "kml2": "http://earth.google.com/kml/2.1",
    "kml3": "http://earth.google.com/kml/2.2",
}
 
 
def parse_coordinates_text(text):
    """Parsea el texto de <coordinates> KML → lista de (lon, lat)."""
    coords = []
    for token in text.strip().split():
        parts = token.split(",")
        if len(parts) >= 2:
            try:
                lon, lat = float(parts[0]), float(parts[1])
                coords.append((lon, lat))
            except ValueError:
                pass
    return coords
 
 
def parse_kmz(kmz_path):
    """
    Lee un archivo KMZ y extrae todos los LineString como objetos Shapely.
    Retorna: (lista de LineString, lista de nombres)
    """
    print(f"\n[KMZ] Leyendo: {kmz_path}")
    lines  = []
    names  = []
 
    try:
        with zipfile.ZipFile(kmz_path, "r") as kmz:
            kml_files = [f for f in kmz.namelist() if f.lower().endswith(".kml")]
            if not kml_files:
                print("[KMZ] ✗ No se encontró archivo .kml dentro del KMZ")
                return [], []
 
            for kml_file in kml_files:
                print(f"[KMZ] Procesando: {kml_file}")
                with kmz.open(kml_file) as kf:
                    tree = etree.parse(kf)
                    root = tree.getroot()
 
                # Detectar namespace automáticamente
                tag = root.tag
                ns  = tag[1:tag.index("}")] if "}" in tag else ""
                ns_map = {"k": ns} if ns else {}
 
                prefix = "{" + ns + "}" if ns else ""
 
                # Buscar placemarks con nombre
                placemark_tag    = f"{prefix}Placemark"
                name_tag         = f"{prefix}name"
                linestring_tag   = f"{prefix}LineString"
                coordinates_tag  = f"{prefix}coordinates"
                multi_geo_tag    = f"{prefix}MultiGeometry"
 
                for pm in root.iter(placemark_tag):
                    name_el = pm.find(name_tag)
                    pm_name = name_el.text.strip() if name_el is not None and name_el.text else "Sin nombre"
 
                    for ls in pm.iter(linestring_tag):
                        coord_el = ls.find(coordinates_tag)
                        if coord_el is not None and coord_el.text:
                            coords = parse_coordinates_text(coord_el.text)
                            if len(coords) >= 2:
                                lines.append(LineString(coords))
                                names.append(pm_name)
 
        print(f"[KMZ] ✓ {len(lines)} líneas extraídas")
        return lines, names
 
    except FileNotFoundError:
        print(f"[KMZ] ✗ Archivo no encontrado: {kmz_path}")
        return [], []
    except Exception as e:
        print(f"[KMZ] ✗ Error: {e}")
        return [], []

In [23]:
# ─────────────────────────────────────────────
# 5.  ANÁLISIS DE PROXIMIDAD
# ─────────────────────────────────────────────
 
# Factor de conversión aproximado grados → metros en Colombia
DEG_TO_M = 111_000  # ~111 km por grado
 
 
def degrees_to_meters(deg):
    return deg * DEG_TO_M
 
 
def analyze_proximity(hazards, kmz_lines, kmz_names):
    """
    Para cada hazard calcula:
      - distancia_m: distancia al LineString más cercano
      - nearest_line: nombre de la línea más cercana
      - closest_point_on_line: punto exacto en la línea
    Retorna la lista de hazards enriquecidos con estos campos.
    """
    if not kmz_lines:
        for h in hazards:
            h["_dist_m"]   = None
            h["_line_name"] = "N/A"
        return hazards
 
    kmz_union = unary_union(kmz_lines)
 
    for h in hazards:
        lat = h.get("location", {}).get("y") #h["latitude"]
        lon = h.get("location", {}).get("x") #h["longitude"]
        pt  = Point(lon, lat)
 
        # Distancia al conjunto de líneas
        dist_deg     = kmz_union.distance(pt)
        h["_dist_m"] = round(degrees_to_meters(dist_deg))
 
        # Línea individual más cercana
        min_dist  = float("inf")
        min_name  = "N/A"
        min_point = None
        for line, name in zip(kmz_lines, kmz_names):
            d = line.distance(pt)
            if d < min_dist:
                min_dist  = d
                min_name  = name
                # Punto exacto en la línea
                _, nearest = nearest_points(pt, line)
                min_point  = nearest
 
        h["_line_name"]   = min_name
        h["_nearest_pt"]  = min_point  # shapely Point (lon, lat)
 
    # Ordenar por distancia
    hazards.sort(key=lambda h: h["_dist_m"] if h["_dist_m"] is not None else 9e9)
    return hazards

In [25]:
# ─────────────────────────────────────────────
# 6.  MAPA FOLIUM
# ─────────────────────────────────────────────
 
def ts_to_str(millis):
    try:
        return datetime.fromtimestamp(millis / 1000).strftime("%Y-%m-%d %H:%M")
    except Exception:
        return "—"
 
 
def create_folium_map(center_lat, center_lon, hazards, kmz_lines, kmz_names,
                      buffer_m=300):
    """Crea el mapa interactivo con todos los elementos."""
 
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=13,
        tiles="CartoDB positron",
    )
 
    # ── Tile layers adicionales ──────────────────────────
    folium.TileLayer("OpenStreetMap",      name="OpenStreetMap").add_to(m)
    folium.TileLayer("CartoDB dark_matter", name="Dark (noche)").add_to(m)
 
    # ── Grupos de capas ─────────────────────────────────
    grp_kmz     = folium.FeatureGroup(name="🛣️ Líneas KMZ",          show=True)
    grp_hazards = folium.FeatureGroup(name="⚠️ Obras en vía (Waze)", show=True)
    grp_buffer  = folium.FeatureGroup(name=f"🔵 Área influencia ({buffer_m}m)", show=True)
    grp_links   = folium.FeatureGroup(name="🔗 Línea al punto más cercano", show=True)
 
    # ── 1. Líneas KMZ ────────────────────────────────────
    colors_kmz = ["#0066ff", "#ff6600", "#009933", "#9900cc", "#cc0000"]
    for i, (line, name) in enumerate(zip(kmz_lines, kmz_names)):
        coords_ll = [(lat, lon) for lon, lat in line.coords]  # flip a (lat, lon)
        color = colors_kmz[i % len(colors_kmz)]
        folium.PolyLine(
            coords_ll,
            color=color,
            weight=5,
            opacity=0.85,
            tooltip=f"<b>{name}</b>",
            popup=folium.Popup(
                f"<b>Vía:</b> {name}<br>"
                f"<b>Segmentos:</b> {len(line.coords) - 1}<br>"
                f"<b>Longitud aprox:</b> {degrees_to_meters(line.length):.0f} m",
                max_width=250,
            ),
        ).add_to(grp_kmz)
 
    # ── 2. Hazards + buffer + línea al KMZ más cercano ──
    is_mock = any("MOCK" in str(h.get("uuid", "")) for h in hazards)
 
    for i, h in enumerate(hazards):
        lat = h.get("location", {}).get("y") #h["latitude"]
        lon = h.get("location", {}).get("x") #h["longitude"]
        dist_m    = h.get("_dist_m")
        line_name = h.get("_line_name", "N/A")
        nearest   = h.get("_nearest_pt")
        street    = h.get("street", "—") or "—"
        city      = h.get("city",   "—") or "—"
        reliability = h.get("reliability", "—")#h.get("alert_reliability", "—")
        thumbs_up   = h.get("reportRating", 0)#h.get("num_thumbs_up", 0)
        pub_millis  = h.get("fecha", 0)
        rank        = i + 1  # ya ordenados por distancia
 
        # Color según distancia
        if dist_m is None:
            icon_color = "gray"
        elif dist_m <= 100:
            icon_color = "red"
        elif dist_m <= 500:
            icon_color = "orange"
        else:
            icon_color = "blue"
 
        dist_label = f"{dist_m:,} m" if dist_m is not None else "N/A"
        mock_badge = " [DEMO]" if is_mock else ""
 
        popup_html = f"""
        <div style="font-family:Arial;font-size:13px;min-width:240px">
          <h4 style="margin:0;color:#d35400">
            ⚠️ OBRA EN VÍA{mock_badge} — #{rank}
          </h4>
          <hr style="margin:6px 0">
          <b>Calle:</b> {street}<br>
          <b>Ciudad:</b> {city}<br>
          <b>Confiabilidad:</b> {reliability}/10 &nbsp;👍 {thumbs_up}<br>
          <b>Reportado:</b> {ts_to_str(pub_millis)}<br>
          <hr style="margin:6px 0">
          <b>📏 Distancia a vía KMZ:</b> <span style="color:#2980b9"><b>{dist_label}</b></span><br>
          <b>🛣️ Vía más cercana:</b> {line_name}<br>
          <hr style="margin:6px 0">
          <small style="color:#888">UUID: {h.get('uuid','—')}</small>
        </div>
        """
 
        # Buffer (área de influencia)
        folium.Circle(
            location=[lat, lon],
            radius=buffer_m,
            color=icon_color,
            fill=True,
            fill_color=icon_color,
            fill_opacity=0.12,
            weight=1.5,
        ).add_to(grp_buffer)
 
        # Marcador
        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color=icon_color, icon="warning-sign", prefix="glyphicon"),
            tooltip=f"⚠️ {street} | {dist_label} de '{line_name}'",
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(grp_hazards)
 
        # Línea punteada al punto más cercano del KMZ
        if nearest is not None:
            folium.PolyLine(
                locations=[[lat, lon], [nearest.y, nearest.x]],
                color="#e74c3c",
                weight=1.5,
                dash_array="6 4",
                opacity=0.7,
                tooltip=f"Dist: {dist_label}",
            ).add_to(grp_links)
            # Punto exacto de conexión
            folium.CircleMarker(
                location=[nearest.y, nearest.x],
                radius=4,
                color="#e74c3c",
                fill=True,
                fill_color="white",
                fill_opacity=1,
                weight=2,
            ).add_to(grp_links)
 
    # Agregar grupos al mapa
    grp_kmz.add_to(m)
    grp_buffer.add_to(m)
    grp_links.add_to(m)
    grp_hazards.add_to(m)
 
    # ── Marcador de centro de búsqueda ──────────────────
    folium.Marker(
        location=[center_lat, center_lon],
        icon=folium.Icon(color="green", icon="home", prefix="glyphicon"),
        tooltip="Centro de búsqueda",
    ).add_to(m)
 
    folium.Circle(
        location=[center_lat, center_lon],
        radius=CONFIG["search_radius_km"] * 1000,
        color="#27ae60",
        fill=False,
        weight=2,
        dash_array="8 4",
        opacity=0.5,
        tooltip=f"Área de búsqueda ({CONFIG['search_radius_km']} km)",
    ).add_to(m)
 
    # ── Panel de estadísticas ────────────────────────────
    n_kmz     = len(kmz_lines)
    n_hazards = len(hazards)
    near_100  = sum(1 for h in hazards if (h.get("_dist_m") or 9999) <= 100)
    near_500  = sum(1 for h in hazards if (h.get("_dist_m") or 9999) <= 500)
    avg_dist  = (sum(h["_dist_m"] for h in hazards if h.get("_dist_m") is not None) / max(n_hazards, 1))
    source_label = "⚠️ DATOS DE PRUEBA (MOCK)" if is_mock else "✅ DATOS REALES WAZE"
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
 
    legend_html = f"""
    <div style="
        position: fixed; bottom:30px; left:30px; z-index:1000;
        background:white; border-radius:10px; padding:16px;
        box-shadow:0 4px 20px rgba(0,0,0,0.2);
        font-family:Arial; font-size:13px; min-width:260px;
        border-left: 4px solid #e74c3c;
    ">
      <h4 style="margin:0 0 10px;color:#2c3e50">📊 Resumen MVP</h4>
      <b>Fuente:</b> <span style="color:{'#e74c3c' if is_mock else '#27ae60'}">{source_label}</span><br>
      <b>Actualizado:</b> {ts}<br>
      <hr style="margin:8px 0">
      <b>🛣️ Líneas KMZ:</b> {n_kmz}<br>
      <b>⚠️ Obras detectadas:</b> {n_hazards}<br>
      <b>🔴 ≤ 100m de vía:</b> {near_100}<br>
      <b>🟠 ≤ 500m de vía:</b> {near_500}<br>
      <b>📏 Distancia promedio:</b> {avg_dist:,.0f} m<br>
      <b>🔵 Buffer activo:</b> {buffer_m} m<br>
      <hr style="margin:8px 0">
      <small style="color:#888">
        Radio búsqueda: {CONFIG["search_radius_km"]} km<br>
        Centro: {center_lat:.4f}, {center_lon:.4f}
      </small>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
 
    # ── Leyenda de colores ───────────────────────────────
    color_legend = """
    <div style="
        position:fixed; bottom:30px; right:30px; z-index:1000;
        background:white; border-radius:10px; padding:14px;
        box-shadow:0 4px 20px rgba(0,0,0,0.2);
        font-family:Arial; font-size:12px;
    ">
      <b>Distancia a vía KMZ</b><br>
      <span style="color:#e74c3c">●</span> ≤ 100 m (crítico)<br>
      <span style="color:#e67e22">●</span> ≤ 500 m (cercano)<br>
      <span style="color:#2980b9">●</span> &gt; 500 m (lejano)<br>
      <hr style="margin:6px 0">
      <span style="color:#0066ff">━━</span> Línea KMZ<br>
      <span style="color:#e74c3c">- -</span> Enlace a punto cercano
    </div>
    """
    m.get_root().html.add_child(folium.Element(color_legend))
 
    # ── Control de capas ─────────────────────────────────
    folium.LayerControl(collapsed=False).add_to(m)
 
    return m

In [29]:
    # ── Main temporal debo corregirlo ──────────────────

print("=" * 60)
print("   WAZE MVP — HAZARD_ON_ROAD_CONSTRUCTION Detector")
print("=" * 60)
 
center   = f"{CONFIG["center_lat"]},{CONFIG["center_lon"]}"
r_km  = CONFIG["search_radius_km"]
buf_m = CONFIG["buffer_radius_m"]
 
# ── Paso 1: Consultar Waze y guardar info ────────────────────────────
#waze_data = fetch_waze_alerts(center, r_km, url)
waze_data  = data
file_path = save_response(waze_data)

# ── Paso 2: Filtrar obras ─────────────────────────────
#hazards = filter_construction_hazards(waze_data) #no lo hago aun solo por ver el mapa final
#hazards = waze_data.get("data", {}).get("alerts", [])
hazards = waze_data.get("alerts", [])

if not hazards:
    print("\n[WARN] No se encontraron eventos HAZARD_ON_ROAD_CONSTRUCTION")
    print("       Generando mapa vacío de todas formas...")

# ── Paso 3: Parsear KMZ ───────────────────────────────
kmz_lines, kmz_names = parse_kmz(CONFIG["kmz_path"])
 
# ── Paso 4: Calcular proximidad ───────────────────────
print("\n[ANÁLISIS] Calculando proximidad a líneas KMZ...")
hazards = analyze_proximity(hazards, kmz_lines, kmz_names)
 
# ── Paso 5: Imprimir ranking ──────────────────────────
print("\n" + "─" * 60)
print(f"  TOP OBRAS más cercanas a líneas KMZ ({len(hazards)} total)")
print("─" * 60)
for i, h in enumerate(hazards[:10], 1):
    dist  = h.get("_dist_m")
    calle = h.get("street", "—") or "—"
    lname = h.get("_line_name", "N/A")
    print(f"  #{i:2d}  {calle[:35]:<35}  →  {dist:>6,} m  de  '{lname}'")
if len(hazards) > 10:
    print(f"       ... y {len(hazards) - 10} más")
print("─" * 60)

# ── Paso 6: Crear mapa ────────────────────────────────
print(f"\n[MAPA] Generando mapa interactivo...")
m = create_folium_map(CONFIG["center_lat"], CONFIG["center_lon"], hazards, kmz_lines, kmz_names, buf_m)
 
output = CONFIG["output_html"]
m.save(output)
print(f"[MAPA] ✓ Guardado: {output}")
print(f"\n  ➜  Abre en tu navegador: {output}\n")
print("=" * 60)

   WAZE MVP — HAZARD_ON_ROAD_CONSTRUCTION Detector
[WAZE] ✓ Guardado en: waze_data_20260501_185114.json

[KMZ] Leyendo: /workspaces/Rup_tub/KMZ/TuberiaAcero_Cartagena.kmz
[KMZ] Procesando: doc.kml
[KMZ] ✓ 217 líneas extraídas

[ANÁLISIS] Calculando proximidad a líneas KMZ...

────────────────────────────────────────────────────────────
  TOP OBRAS más cercanas a líneas KMZ (46 total)
────────────────────────────────────────────────────────────
  # 1  Calle 31 >(N)                        →       3 m  de  '8364136'
  # 2  Diagonal 21 B                        →      25 m  de  '2894551'
  # 3  Calle 20                             →      26 m  de  '11417479'
  # 4  Vía al Mar / RN90A-01 >Occidente     →      33 m  de  '11417479'
  # 5  Av. El Pedregal                      →      47 m  de  '8361729'
  # 6  Carrera 81 A                         →      77 m  de  '908371'
  # 7  Transversal 73                       →      83 m  de  '8384066'
  # 8  Carrera 81 A                         →     106 